# V17 · Netzwerke und Industrie 4.0: JSON, HTTP und IP-Adressen — Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Python-Werkstatt im Labor, dann das Training zu Hause (Browser, ca. 70 Min). Dieses Notebook ist die **Vertiefung** und läuft am besten im **Codespace** oder in Jupyter. Es nutzt Python aus V01 bis V16: Funktionen, Listen, Dictionaries, Dateien, try/except, Bibliotheken, und neu `json`, `requests` und `ipaddress`.

**Der rote Faden:** die Fertigungszelle **FZ-3** in Halle 2 (Drehmaschine D2, Fräsmaschine F1, Roboter R1, Messmaschine M1, Zellensteuerung SPS). Ihre Daten liegen als JSON auf der Kurs-Website. Im Browser-Labor hast du sie mit einer relativen Adresse geholt (`../assets/...`). Hier im Notebook läuft Python **außerhalb des Browsers**, deshalb steht die volle Internet-Adresse in der Variable `URL`. Du brauchst eine Internetverbindung.

**Der Arbeitsablauf bei jeder Aufgabe:** (1) Die Daten ansehen (`print`), bevor du sie auswertest. (2) Von außen nach innen zugreifen, ein Schlüssel oder Index pro Ebene. (3) Bei Daten aus dem Netz zuerst den Status prüfen. (4) Die Prüfzelle laufen lassen. Die Prüfzellen sind keine Musterlösungen, sie zeigen nur, ob dein Ergebnis stimmt.

**Stufen:** Aufwärmen (1.x), Anwenden (2.x), Knobeln (3.x). Lösungen gibt es hier nicht. Tipps sind eingeklappt.

In [ ]:
# Werkzeug-Zelle für die Selbsttests: einfach einmal ausführen, nicht verändern.
def pruefe(ist, soll, text):
    if ist == soll:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")


def pruefe_ja(bedingung, text):
    if bedingung:
        print("OK: " + text)
    else:
        print("FEHLER: " + text)


# Adresse der Zellen-Daten (statische JSON-Dateien auf der Kurs-Website)
URL = "https://nice-wave-051f81f03.3.azurestaticapps.net/assets/data/v17/api/"
print("Bereit. Daten unter:", URL)

## Stufe 1 — Aufwärmen

### 1.1 JSON-Text lesen
Eine Maschine meldet ihre Daten als JSON-**Text**. Wandle ihn mit `json.loads` in ein Dictionary um. Lege dann `programm` (der Wert zum Schlüssel `"programm"`) und `drehzahl` (zu `"drehzahl_1_min"`) an.

Merke: `loads` mit **s** liest Text, `load` ohne s liest eine Datei.

<details><summary>Tipp anzeigen</summary>

Tipp: `d = json.loads(text)`, danach `d["programm"]` und `d["drehzahl_1_min"]`. Die Zahl bleibt eine Zahl (`int`).
</details>

In [ ]:
import json

text = '{"id": "F1", "status": "laeuft", "programm": "FLANSCH_80", "drehzahl_1_min": 3000, "kuehlung": true, "fehler": null}'

# TODO: Text in ein Dictionary umwandeln (json.loads), dann zwei Werte herausholen
d = None
programm = None
drehzahl = None
print(programm, drehzahl)

In [ ]:
pruefe(programm, "FLANSCH_80", "Programm gelesen")
pruefe(drehzahl, 3000, "Drehzahl gelesen")
pruefe(type(drehzahl), int, "Die Drehzahl ist eine Zahl (int), kein Text")
pruefe(d["kuehlung"] if isinstance(d, dict) else None, True, "JSON true ist in Python True")
pruefe(d["fehler"] if isinstance(d, dict) else 1, None, "JSON null ist in Python None")

### 1.2 Verschachtelt zugreifen
Die Zelle ist ein Dictionary mit einer **Liste von Dictionaries** und einem **Dictionary im Dictionary**. Hole die drei Werte. Du liest von außen nach innen: Liste = Nummer, Dictionary = Schlüssel.

<details><summary>Tipp anzeigen</summary>

Tipp: Der Roboter ist das dritte Element, also Index 2: `zelle["maschinen"][2]["id"]`. Der Verschleiß: `zelle["verschleiss"]["T4"]`. Für einen fehlenden Schlüssel nimmst du `.get("T9", 0)`.
</details>

In [ ]:
zelle = {
    "zelle": "FZ-3",
    "maschinen": [
        {"id": "D2", "status": "laeuft"},
        {"id": "F1", "status": "laeuft"},
        {"id": "R1", "status": "gestoert"},
        {"id": "M1", "status": "bereit"},
    ],
    "verschleiss": {"T1": 22, "T4": 83},
}

# TODO: ID der dritten Maschine, Verschleiß von T4, Verschleiß von T9 (gibt es nicht: Ersatzwert 0)
dritte_id = None
t4 = None
t9 = None
print(dritte_id, t4, t9)

In [ ]:
pruefe(dritte_id, "R1", "ID der dritten Maschine")
pruefe(t4, 83, "Verschleiß T4")
pruefe(t9, 0, "T9 fehlt, Ersatzwert 0 (mit .get)")

### 1.3 Dictionary in JSON-Text
Baue die Wartungsmeldung für den MQTT-Broker: Maschine D2, Werkzeug T4, Aktion `wechseln`. Wandle sie mit `json.dumps` in Text um (`dumps` mit s: Dictionary zu Text).

In [ ]:
import json

# TODO: Dictionary mit den Schlüsseln maschine, werkzeug, aktion bauen und als Text in "nutzlast" ablegen
nutzlast = None
print(nutzlast)

In [ ]:
pruefe(isinstance(nutzlast, str), True, "nutzlast ist ein Text")
pruefe(json.loads(nutzlast) if isinstance(nutzlast, str) else None, {"maschine": "D2", "werkzeug": "T4", "aktion": "wechseln"}, "Der Text enthält die drei Angaben als gültiges JSON")

## Stufe 2 — Anwenden

### 2.1 Die Drehmaschine abfragen
Hole `D2.json` mit `requests`. **Prüfe zuerst den Status** (`200` heißt „in Ordnung“). Lege danach `programm` und eine Liste `zu_wechseln` an: alle Werkzeuge mit Verschleiß über 80 %.

Schau dir die Antwort zuerst mit `print(daten)` an, damit du die Schlüssel kennst.

<details><summary>Tipp anzeigen</summary>

Tipp: `antwort = requests.get(URL + "D2.json")`, `antwort.status_code`, `daten = antwort.json()`. Der Verschleiß steht in `daten["werkzeugverschleiss_prozent"]`, ein Dictionary Werkzeug → Prozent. Mit `.items()` bekommst du beide Teile in der Schleife.
</details>

In [ ]:
import requests

antwort = requests.get(URL + "D2.json")
status = antwort.status_code
print("Status:", status)

programm = None
zu_wechseln = []
# TODO: nur wenn der Status 200 ist: Daten lesen, Programm merken, Werkzeuge über 80 % in die Liste
print(programm, zu_wechseln)

In [ ]:
pruefe(status, 200, "Anfrage erfolgreich")
pruefe(programm, "WELLE_20H6", "Programm der Drehmaschine")
pruefe(zu_wechseln, ["T4"], "Nur T4 liegt über 80 %")

### 2.2 Geräte im falschen Netz finden
Hole `maschinen.json` und prüfe mit `ipaddress`, welche Geräte **nicht** im Zellennetz liegen (das Netz steht in den Daten unter `"netz"`). Lege die IDs in die Liste `falsch`. Dein Ergebnis zeigt, warum der Leitstand einen Roboter „nicht erreicht“.

<details><summary>Tipp anzeigen</summary>

Tipp: `netz = ipaddress.ip_network(zelle["netz"])`, in der Schleife `ipaddress.ip_address(m["ip"])` und der Operator `in` (oder `not in`). Ein Textvergleich mit `startswith` wäre falsch, siehe Code-Review im Labor.
</details>

In [ ]:
import ipaddress
import requests

antwort = requests.get(URL + "maschinen.json")
zelle = antwort.json()
print("Netz:", zelle["netz"], "| Gateway:", zelle["gateway"])

falsch = []
# TODO: für jede Maschine prüfen, ob ihre IP im Netz liegt; falls nicht, die ID in falsch aufnehmen
print(falsch)

In [ ]:
pruefe(falsch, ["R1"], "Nur der Roboter R1 liegt im falschen Netz")
pruefe(ipaddress.ip_network(zelle["netz"]).num_addresses - 2, 254, "Ein /24-Netz hat 254 nutzbare Adressen")

### 2.3 Fehler abfangen: kaputter Mitschnitt
Ein MQTT-Client hat vier Zeilen mitgeschrieben, **eine davon ist kaputt**. Lies jede Zeile mit `json.loads`, fange den `ValueError` ab und zähle die gültigen Nachrichten (`gueltig`) und die kaputten (`kaputt`). Wiederholung von V09: gezielt abfangen, nie ein leeres `except:`.

In [ ]:
import json

zeilen = [
    '{"topic": "fz3/D2/status", "nutzlast": {"status": "laeuft"}}',
    '{"topic": "fz3/F1/status", "nutzlast": {"status": "laeuft"',
    '{"topic": "fz3/R1/status", "nutzlast": {"status": "gestoert"}}',
    '{"topic": "fz3/M1/status", "nutzlast": {"status": "bereit"}}',
]

gueltig = 0
kaputt = 0
# TODO: Schleife über zeilen, try/except ValueError
print(gueltig, kaputt)

In [ ]:
pruefe(gueltig, 3, "Drei gültige Nachrichten")
pruefe(kaputt, 1, "Eine kaputte Zeile übersprungen")

## Stufe 3 — Knobeln

### 3.1 Der Zellenbericht
Rechne die **Stückzahl der Schicht** der ganzen Zelle zusammen: Hole `maschinen.json`, dann für **jedes** Gerät die Datei `ID + ".json"`, und addiere `stueckzahl_schicht`. Hat ein Gerät keinen solchen Schlüssel (die SPS), zählt es als 0. Packe die Rechnung in eine Funktion `schichtsumme(url)`, damit du sie mit einer anderen Zelle wiederverwenden kannst.

<details><summary>Tipp anzeigen</summary>

Tipp: Pro Gerät eine zweite Anfrage: `requests.get(url + m["id"] + ".json").json()`. Dann `details.get("stueckzahl_schicht", 0)`. Die Funktion gibt die Summe mit `return` zurück.
</details>

In [ ]:
import requests


def schichtsumme(url):
    # TODO: Geräteliste holen, jedes Gerät abfragen, Stückzahlen addieren
    return None


summe = schichtsumme(URL)
print("Stückzahl der Schicht:", summe)

In [ ]:
pruefe(summe, 546, "Gesamtstückzahl der Schicht (142 + 61 + 203 + 140 + 0)")

### 3.2 Netzprüfung ohne Bibliothek
Schreibe `falsches_netz(zelle)` **ohne** `ipaddress`: Es gibt die IDs der Geräte zurück, deren erste **drei Blöcke** der IP nicht mit denen des Netzes übereinstimmen (nur für /24-Netze richtig). Nutze `split(".")` und `[:3]`. Probiere es zuerst auf Papier, wie in der Klausur.

Danach die Frage zum Nachdenken: Was geht bei dieser Abkürzung schief, wenn das Netz ein /26 ist?

<details><summary>Tipp anzeigen</summary>

Tipp: `zelle["netz"].split("/")[0].split(".")[:3]` ergibt die Liste der drei Blöcke des Netzes. Genauso für `m["ip"]`. Zwei Listen vergleichst du mit `!=`.
</details>

In [ ]:
def falsches_netz(zelle):
    # TODO
    return None


print(falsches_netz(zelle))

In [ ]:
pruefe(falsches_netz(zelle), ["R1"], "FZ-3: nur R1 im falschen Netz")
anders = {"netz": "10.1.5.0/24", "maschinen": [{"id": "A", "ip": "10.1.5.20"}, {"id": "B", "ip": "10.1.6.20"}]}
pruefe(falsches_netz(anders), ["B"], "Auch mit einem anderen Netz")
pruefe(falsches_netz({"netz": "192.168.30.0/24", "maschinen": []}), [], "Ohne Geräte: leere Liste")

### 3.3 Eigene Frage an die Daten (frei)
Wähle selbst: Welche Maschine hat die höchste Spindeltemperatur? Welches Gerät hat **keinen** `werkzeugverschleiss_prozent`? Wie viele Maschinen sind `laeuft`? Hole die Daten, lies sie mit Schlüsseln und beantworte die Frage mit einem `print`. Es gibt keine Prüfzelle. Zeige dein Ergebnis einem Mitstudierenden und lass dir erklären, ob dein Zugriff auch bei fehlendem Schlüssel hält.

In [ ]:
# Dein Code

## Ausblick: MQTT mit Python (freiwillig, nicht ausführbar im Browser)
In der Lerneinheit hast du gesehen, wie Maschinen ihren Status per **MQTT** auf Topics wie `fz3/D2/status` veröffentlichen und der Leitstand `fz3/+/status` abonniert. Mit der Bibliothek **paho-mqtt** sieht ein Abonnent so aus. Das läuft nur mit einem erreichbaren Broker (in der Fabrik `192.168.30.5:1883`; zum Üben z. B. ein lokaler Mosquitto-Broker im Codespace):

```python
# pip install paho-mqtt   (Version 2.x)
import json
import paho.mqtt.client as mqtt


def bei_nachricht(client, userdata, nachricht):
    daten = json.loads(nachricht.payload)          # Nutzlast: JSON-Text -> Dictionary
    print(nachricht.topic, daten)


client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_message = bei_nachricht
client.connect("192.168.30.5", 1883)
client.subscribe("fz3/+/status")                   # + = genau eine Ebene
client.loop_forever()
```

Du erkennst die Bausteine wieder: `json.loads` für die Nutzlast, ein Topic mit Platzhalter, ein Programmteil, der bei jeder Nachricht aufgerufen wird. **HTTP** holt Daten, wenn *du* fragst (Pull). **MQTT** meldet sich, wenn sich *etwas ändert* (Push). Wie man einen Broker absichert (Zugangsdaten, Verschlüsselung), delegierst du an die IT: Das ist nicht deine Aufgabe als Ingenieur, du musst nur wissen, dass es sein muss.

## Selbsttest: Kannst du es im Kopf?
Beantworte erst auf Papier, dann prüfe im Notebook.

1. Liegt `192.168.31.20` im Netz `192.168.30.0/24`? Wie viele nutzbare Adressen hat ein `/26`-Netz?
2. Gegeben `d = json.loads('{"w": {"T4": 83}, "z": [61, 12]}')`: Was ist `d["w"]["T4"]`, was ist `d["z"][1]`?
3. Was bedeutet der Statuscode 404? Was musst du tun, bevor du `.json()` aufrufst?
4. Welche Topics empfängt ein Abonnent von `fz3/+/status`?

In [ ]:
import ipaddress

print(ipaddress.ip_address("192.168.31.20") in ipaddress.ip_network("192.168.30.0/24"))   # 1
print(ipaddress.ip_network("192.168.30.0/26").num_addresses - 2)                          # 1
d = json.loads('{"w": {"T4": 83}, "z": [61, 12]}')
print(d["w"]["T4"], d["z"][1])                                                              # 2